In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from tqdm import tqdm
import json
from mtrain.neg_mask.leveled_cropping import load_crop_level_sample_from_directory
from mtrain.neg_mask.model.datasets.foviate_shrink import get_foviated_image_and_mask
from mtrain.neg_mask.crops import get_region_crops, bbox_only_mask, get_largest_bbox
import albumentations as A
from mtrain.utils import *
from mtrain.smallnet.unet.extract.draw import show_extracted_dataset
import shutil
from mtrain.seg import mapillary as mapi

In [ ]:
from mtrain.example_dir import ExampleDir
from mtrain.smallnet.unet.extract.taco_to_fastai import extract_taco_dataset
from pycocotools.coco import COCO
from mtrain.neg_mask.walls import get_trash_mask_regions_fully_enclosed_in_mapi_region
import shutil

# commons


In [ ]:
from mtrain.neg_mask.crops import Bbox
from dataclasses import dataclass, asdict

NEGMASK_DATASET_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training"
)


@dataclass
class ModelResult:
    bbox: Bbox
    label: str

    @classmethod
    def from_json(cls, json_path) -> "ModelResult":
        with open(json_path) as f:
            mres = json.load(f)
            bb = mres["bbox"]
            bb = Bbox(x=bb["x"], y=bb["y"], w=bb["w"], h=bb["h"])
            mres = ModelResult(bbox=bb, label=mres["label"])
            return mres

In [ ]:
def show_l1_dataset(ds_root, num_samples=4):
    dirs = globL(ds_root, "*")
    dirs = random.sample(dirs, num_samples)
    sms = [
        (DiskImage.load(d / "image.jpg"), DiskBooleanMask.load(d / "mask.png"))
        for d in dirs
    ]
    show(it_chain(sms))


def get_data_from_l1_dir(single_example_l1_dir):
    d = single_example_l1_dir
    image = DiskImage.load(d / "image.jpg")
    mask = DiskBooleanMask.load(d / "mask.png")
    with open(d / "model.json") as f:
        mres = json.load(f)
        bb = mres["bbox"]
        bb = Bbox(x=bb["x"], y=bb["y"], w=bb["w"], h=bb["h"])
        mres = ModelResult(bbox=bb, label=mres["label"])
    return image, mask, mres

In [ ]:
show_l1_dataset(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/walls-mapillary/L1"))

# L1

This dataset has all the data in high resolution (original resolution of the image).  
The only thing is that each image and mask is a single example (mask has a single region also). We will also add the bbox in `bbox.json`

## Manually annotated

In [ ]:
SRC_CROP_LEVEL_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level"
)
DEST_CROP_LEVEL_DIR = mkdir(NEGMASK_DATASET_DIR / "manually_annotated" / "L1")

In [ ]:
for label in ["other", "trash"]:
    dirs = list((SRC_CROP_LEVEL_DIR / label).glob("*"))
    for p in tqdm(dirs):
        if (
            not p.is_dir()
            or not (p / "image.jpg").exists()
            or not (p / "source_dir" / "image.jpg").exists()
        ):
            continue

        try:
            sample = load_crop_level_sample_from_directory(p)
        except Exception as ex:
            print(f"WARN: failed in loading sample at {p.name} cause={ex}")
            continue
        img, mask = sample.full_image, sample.full_mask
        bbox = sample.bbox

        dest_dir = mkdir(DEST_CROP_LEVEL_DIR / p.name)

        DiskImage.save(img, dest_dir / "image.jpg")
        DiskBooleanMask.save(mask, dest_dir / "mask.png")

        res_bbox = Bbox(int(bbox.x), int(bbox.y), int(bbox.w), int(bbox.h))
        mres = ModelResult(bbox=res_bbox, label=label)
        mres = asdict(mres)
        with open(dest_dir / "model.json", "w") as f:
            json.dump(mres, f)

In [ ]:
show_l1_dataset(DEST_CROP_LEVEL_DIR, 4)

## TACO

In [ ]:
TACO_DIR = Path("/Users/hariomnarang/Desktop/personal/TACO/data")
ANN_FILE = TACO_DIR / "annotations.json"


OUT_DIR = mkdir(NEGMASK_DATASET_DIR / "taco" / "preprocess")
BIN_OUT = mkdir(OUT_DIR / "binary")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_IMG_ID = 342
img_id = TEST_IMG_ID
coco = COCO(ANN_FILE)

USEFUL_CATS_DIR = OUT_DIR / "useful-cats-ext"


### coco -> image+mask

In [ ]:
CIGARETTE = 59
PLASTIC_FILM = 36
OTHER_PLASTIC = 29
OTHER_PLASTIC_WRAPPER = 39
DISPOSABLE_PLASTIC_CUP = 21
NORMAL_PAPER = 33
PLASTIC_LID = 27
PAPER_CUP = 20
PLASTIC_UTENSILS = 49
CRISP_PACKET = 42
PLASTIC_STRAW = 55
CLEAR_PLASTIC_BOTTLE = 5
WRAPPING_PAPER = 32
MAGAZINE_PAPER = 30
PAPER_BAG = 34
GARBAGE_BAG = 38
TISSUE_PAPER = 31
SINGLE_USE_CARRIER_BAG = 40

# CIGARETTE,
useful_cats = [
    CIGARETTE,
    PLASTIC_FILM,
    OTHER_PLASTIC,
    OTHER_PLASTIC_WRAPPER,
    DISPOSABLE_PLASTIC_CUP,
    NORMAL_PAPER,
    PLASTIC_LID,
    PAPER_CUP,
    PLASTIC_UTENSILS,
    CRISP_PACKET,
    CLEAR_PLASTIC_BOTTLE,
    WRAPPING_PAPER,
    MAGAZINE_PAPER,
    PAPER_BAG,
    GARBAGE_BAG,
    TISSUE_PAPER,
    SINGLE_USE_CARRIER_BAG,
]


In [ ]:
extract_taco_dataset(
    ann_file=ANN_FILE,
    taco_dir=TACO_DIR,
    out_dir=USEFUL_CATS_DIR,
    should_collapse_mask_to_binary=True,
)

In [ ]:
show_extracted_dataset(USEFUL_CATS_DIR, 4)

### L1

In [ ]:
c0 = 0
to_rm = []
to_add = []
for mask_path in tqdm(globL(USEFUL_CATS_DIR / "masks", "*.png")):
    img_path = USEFUL_CATS_DIR / "images" / f"{mask_path.stem}.jpeg"
    m = DiskBooleanMask.load(mask_path)
    if m.sum() == 0:
        to_rm.append((mask_path, img_path))
    else:
        to_add.append((mask_path, img_path))

In [ ]:
len(to_add), len(to_rm)

In [ ]:
def get_image_and_masks_for_single_pair(mask_path, image_path):
    mask = DiskBooleanMask.load(mask_path)
    image = DiskImage.load(image_path)

    bboxes = list(get_region_crops(mask))
    masks = []
    for bbox in bboxes:
        masks.append(bbox_only_mask(mask, bbox, -1))
    return image, masks, bboxes


def save_single_pair(mask_path, image_path, root_dest_dir):
    image, masks, bboxes = get_image_and_masks_for_single_pair(mask_path, image_path)

    for i, mask in enumerate(masks):
        bbox = bboxes[i]
        fname = f"{image_path.stem}_{i}"
        dest_dir = mkdir(root_dest_dir / fname)

        DiskImage.save(image, dest_dir / "image.jpg")
        DiskBooleanMask.save(mask, dest_dir / "mask.png")

        res_bbox = Bbox(int(bbox.x), int(bbox.y), int(bbox.w), int(bbox.h))
        mres = ModelResult(bbox=res_bbox, label="trash")
        mres = asdict(mres)
        with open(dest_dir / "model.json", "w") as f:
            json.dump(mres, f)

In [ ]:
from tqdm import tqdm

DEST_DIR = mkdir(NEGMASK_DATASET_DIR / "taco" / "L1")
for mask_path, image_path in tqdm(to_add):
    save_single_pair(mask_path, image_path, DEST_DIR)

In [ ]:
show_l1_dataset(DEST_DIR)

## Walls and buildings

In [ ]:
def save_to_l1(edir: ExampleDir, l1_dest_dir: Path):
    # we want the relative path to be symlinked
    a = edir.load_all_assets("md", "md")
    image = edir.load_and_resize_image(edir.image_path)
    mapi_mask = mapi.get_mask_with_labels(a['mapi_pred'], [mapi.Label.WALL, mapi.Label.BUILDING])
    regions = get_trash_mask_regions_fully_enclosed_in_mapi_region(a['mask'], mapi_mask)
    for i, (region, bbox) in enumerate(regions):
        dest = mkdir(l1_dest_dir / f"{edir.d.name}-{i}")
        DiskImage.save(image, dest / "image.jpg")
        DiskBooleanMask.save(region, dest / "mask.png")


In [ ]:
WALLS_MAPILLARY = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/walls-mapillary")
RAW_DATA = WALLS_MAPILLARY / "raw_data"
WALLS_L1 = WALLS_MAPILLARY / "L1"
UNSAMPLED_WALLS_L1 = WALLS_MAPILLARY / "unsampled-L1"

In [ ]:
# dirs = globL(RAW_DATA, "*")
# edirs = [ExampleDir(d, {}, {}) for d in dirs]
# for edir in tqdm(edirs):
#     save_to_l1(edir, UNSAMPLED_WALLS_L1)

In [ ]:
show_l1_dataset(UNSAMPLED_WALLS_L1)

In [ ]:
res = []
for d in tqdm(globL(UNSAMPLED_WALLS_L1, "*")):
    area = DiskBooleanMask.load(d / "mask.png").sum()
    res.append((area, d))

In [ ]:
import pandas as pd

def binned_sample(tuples, n_bins, samples_per_bin):
    if not tuples:
        return []
    df = pd.DataFrame(tuples, columns=['area', 'name', 'path'])
    df['area_bin'] = pd.cut(df['area'], n_bins, labels=False)
    return df.groupby('area_bin', group_keys=False).apply(lambda x: x.sample(min(len(x), samples_per_bin)))


def until_last_dash(name):
    return "-".join(name.split("-")[:-1])


In [ ]:
tuples_small = [(a,until_last_dash(Path(p).name),p) for (a,p) in res if a < 6000]
tuples_big = [(a,until_last_dash(Path(p).name),p) for (a,p) in res if 6000 <= a < 10000]

samples_small = binned_sample(tuples_small, 300, 9)
samples_big = binned_sample(tuples_big, 300, 2)
# samples['area'].hist()
# samples
len(samples_small), len(samples_big)

In [ ]:
samples_small['area'].hist()

In [ ]:
samples_big['area'].hist()

In [ ]:
mkdir(WALLS_L1)
l1_paths = list(samples_small['path']) + list(samples_big['path'])
for p in tqdm(l1_paths):
    dest = WALLS_L1 / p.name
    rel_path = f"../unsampled-L1/{p.name}"
    dest.symlink_to(rel_path)

In [ ]:
show_l1_dataset(WALLS_L1)

# L1 -> foveated

In [ ]:
def l1_to_foveated(l1_dir: Path, full_image_size: int, crop_size: int, bbox_pad: int):
    dirs = globL(l1_dir, "*")
    for d in tqdm(dirs):
        image, mask, mres = get_data_from_l1_dir(d)
        (_, _), (re_img, re_mask), re_bbox, _ = get_foviated_image_and_mask(
            image, mask, mres.bbox, full_image_size, crop_size, bbox_pad
        )
        yield re_img, re_mask, re_bbox, d

In [ ]:
MANUAL_L1_DIR = NEGMASK_DATASET_DIR / "manually_annotated" / "L1"
TACO_L1_DIR = NEGMASK_DATASET_DIR / "taco" / "L1"

In [ ]:
it = l1_to_foveated(MANUAL_L1_DIR, 1024, 224, 3)

In [ ]:
re_img, re_mask, re_bbox, d = next(it)

In [ ]:
print(d)
show([re_img, OV(re_img, re_mask), re_img, OV(re_img, re_bbox)])


## Save manual foveated

In [ ]:
# it = l1_to_foveated(MANUAL_L1_DIR, 1024, 224, 3)

# MANUAL_FOVEATED_DIR = mkdir(MANUAL_L1_DIR.parent / "foveated")
# mkdir(MANUAL_FOVEATED_DIR / "train")
# mkdir(MANUAL_FOVEATED_DIR / "masks")

# for (re_img, re_mask, re_bbox, d) in (it):
#     fname = d.name
#     DiskImage.save(re_img, MANUAL_FOVEATED_DIR / "train" / f"{fname}.jpg")
#     DiskBooleanMask.save(re_mask, MANUAL_FOVEATED_DIR / "masks" / f"{fname}.png")

In [ ]:
def save_l1_to_foveated(src_l1_dir, dest_foveated_dir):
    it = l1_to_foveated(src_l1_dir, 1024, 224, 3)

    mkdir(dest_foveated_dir)
    mkdir(dest_foveated_dir / "train")
    mkdir(dest_foveated_dir / "masks")

    for (re_img, re_mask, re_bbox, d) in (it):
        mres = ModelResult.from_json(d / "model.json")
        fname = d.name
        fname = f"{mres.label}_{fname}"
        DiskImage.save(re_img, dest_foveated_dir / "train" / f"{fname}.jpg")
        DiskBooleanMask.save(re_mask, dest_foveated_dir / "masks" / f"{fname}.png")

In [ ]:
save_l1_to_foveated(MANUAL_L1_DIR, MANUAL_FOVEATED_DIR)

In [ ]:
save_l1_to_foveated(TACO_L1_DIR, TACO_L1_DIR.parent / "foveated")

In [ ]:
save_l1_to_foveated(WALLS_L1, WALLS_L1.parent / "foveated")

In [ ]:
show_negmask_ds(WALLS_L1.parent / "foveated", 4)

In [ ]:
show_negmask_ds(MANUAL_FOVEATED_DIR, 4)

In [ ]:
FOVEATED_TACO_DIR = TACO_L1_DIR.parent / "foveated"

In [ ]:
show_negmask_ds(FOVEATED_TACO_DIR, 4)

In [ ]:
MODEL_DATASET_DIR = NEGMASK_DATASET_DIR / "dataset"
mkdir(MODEL_DATASET_DIR / "train")
mkdir(MODEL_DATASET_DIR / "masks")

In [ ]:
from mtrain.neg_mask.model.datasets.copy_ds import copy_negmask_ds_to_ds
# copy_negmask_ds_to_ds(MANUAL_FOVEATED_DIR, MODEL_DATASET_DIR, "manual")
# copy_negmask_ds_to_ds(, MODEL_DATASET_DIR, "manual")

In [ ]:
for img in globL(FOVEATED_TACO_DIR / "train", "*.jpg"):
    if img.stem.startswith("trash"):
        continue
    new_name = f"trash_{img.stem}"
    mask = FOVEATED_TACO_DIR / "masks" / f"{img.stem}.png"
    shutil.move(img, img.parent / f"{new_name}.jpg")
    shutil.move(mask, mask.parent / f"{new_name}.png")


In [ ]:
copy_negmask_ds_to_ds(FOVEATED_TACO_DIR, MODEL_DATASET_DIR, "taco")

In [ ]:
copy_negmask_ds_to_ds(WALLS_L1.parent / "foveated", MODEL_DATASET_DIR, "mapillary-walls")

In [ ]:
show_negmask_ds(MODEL_DATASET_DIR, 8)